## Paso 0 — Cargamos las 18 tablas ya tipadas

Mismo patrón que los notebooks anteriores: leemos directo los Parquet
del notebook 00.


In [1]:
import pandas as pd
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
ARCHIVOS_PARQUET = sorted(RUTA_INTERIM.glob("*.parquet"))
tablas = {archivo.stem: pd.read_parquet(archivo) for archivo in ARCHIVOS_PARQUET}
assert len(tablas) == 18, f"Se esperaban 18 tablas, se cargaron {len(tablas)}"
print(f"Tablas cargadas: {len(tablas)}")


Tablas cargadas: 18


## Paso 1 — ¿Cuántas líneas Tarifario tienen suficiente info para comparar contra SAP? (§7.2 del plan)

Para comparar el precio cotizado contra el precio de mercado SAP,
necesitamos saber qué LineaNegocio/ClaseViaje usó esa línea — dato que
viene de lineas_solicitud, no de cotizacion_lineas directamente. Ya
sabíamos que esto casi nunca se llena (notebook 01) — aquí confirmamos
cuántas de las 32 líneas Tarifario específicamente tienen ese dato.


In [2]:
cotizacion_lineas = tablas["cotizacion_lineas"]
lineas_solicitud = tablas["lineas_solicitud"]

tarifario = cotizacion_lineas[cotizacion_lineas["fuente"] == "Tarifario"].copy()
tarifario = tarifario.merge(
    lineas_solicitud[["id", "lineaNegocio", "claseViaje", "rutaId"]],
    left_on="lineaSolicitudId", right_on="id", suffixes=("", "_linea")
)

print(f"Líneas con Fuente=Tarifario: {len(cotizacion_lineas[cotizacion_lineas['fuente']=='Tarifario'])}")
con_ambos = tarifario.dropna(subset=["lineaNegocio", "claseViaje"])
print(f"De esas, con LineaNegocio Y ClaseViaje pobladas: {len(con_ambos)}")
print()
print(con_ambos[["id", "lineaNegocio", "claseViaje", "rutaId", "valorFlete"]].to_string())


Líneas con Fuente=Tarifario: 32
De esas, con LineaNegocio Y ClaseViaje pobladas: 0

Empty DataFrame
Columns: [id, lineaNegocio, claseViaje, rutaId, valorFlete]
Index: []


## Paso 1b — Cruce real por ruta normalizada contra SAP (§7.1/§7.2 del plan, resuelto)

En el Paso 1 concluimos que cruzar por `LineaNegocio`/`ClaseViaje` da 0
casos usables. Pero en el notebook 06 (Fase 8) sí resolvimos la ruta con
`rapidfuzz` — volvemos aquí a cerrar el ciclo: para las 32 líneas
`Fuente=Tarifario`, resolvemos su ruta (vía `lineaSolicitudId` →
`lineas_solicitud` → `rutaId` → `catalogo_rutas`) a la zona SAP
correspondiente, y comparamos el precio cotizado contra el rango real
de `tarifa-plaza` para esa zona — **excluyendo primero las tarifas de
"Devolución Contenedor"** (`lineanegocio="6"`): es la tarifa de
reposicionar un contenedor vacío, no de transportar carga, así que
mezclarla en el mismo mínimo/mediana/máximo compara servicios distintos.
Sigue sin filtrar por tipo de vehículo (no tenemos esa traducción), así
que el rango de comparación es más amplio de lo ideal, no una
comparación estrictamente pareja.

In [3]:
import re
import unicodedata
from rapidfuzz import process, fuzz

def limpiar_nombre_ciudad(texto):
    if pd.isna(texto):
        return None
    sin_departamento = re.sub(r"\s*\([^)]*\)\s*$", "", texto).strip()
    sin_tildes = unicodedata.normalize("NFKD", sin_departamento).encode("ascii", "ignore").decode("ascii")
    return sin_tildes.upper()

lineas_solicitud = tablas["lineas_solicitud"]
catalogo_rutas = tablas["catalogo_rutas"]
tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]
historico_sap = tablas["historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy"]

base = cotizacion_lineas.merge(
    lineas_solicitud[["id", "rutaId"]], left_on="lineaSolicitudId", right_on="id", suffixes=("", "_linea")
).merge(
    catalogo_rutas[["id", "origen", "destino"]], left_on="rutaId", right_on="id", suffixes=("", "_ruta")
)
base["origen_limpio"] = base["origen"].apply(limpiar_nombre_ciudad)
base["destino_limpio"] = base["destino"].apply(limpiar_nombre_ciudad)

# mismo vocabulario y umbral que en el notebook 06 (§9) — sorted() para orden determinista
valores_sap_codigos = (
    set(tarifa_plaza["origen"].dropna().unique()) | set(tarifa_plaza["destino"].dropna().unique())
    | set(historico_sap["origen"].dropna().unique()) | set(historico_sap["destino"].dropna().unique())
)
vocabulario_sap = {re.sub(r"^ZT?C_", "", c).replace("_", " "): c for c in sorted(valores_sap_codigos)}
nombres_sap = list(vocabulario_sap.keys())

def resolver_zona_sap(ciudad, umbral=90):
    if ciudad is None:
        return None
    mejor = process.extractOne(ciudad, nombres_sap, scorer=fuzz.WRatio)
    return vocabulario_sap[mejor[0]] if mejor[1] >= umbral else None

base["zona_origen"] = base["origen_limpio"].apply(resolver_zona_sap)
base["zona_destino"] = base["destino_limpio"].apply(resolver_zona_sap)

tarifario_ruta = base[base["fuente"] == "Tarifario"].copy()

# Corrección post-cierre (revisión de calidad): "Oferta de Expedición Devolución
# Contenedor" (lineanegocio="6") es la tarifa de reposicionar un contenedor vacío, no
# de transportar carga — se excluye de los candidatos antes de comparar, para no diluir
# el mínimo/mediana/máximo con un servicio distinto al que de verdad cotizó CabinaFletes.
candidatos_carga_real = tarifa_plaza[tarifa_plaza["lineanegocio"] != "6"]

filas = []
for _, fila in tarifario_ruta.iterrows():
    candidatos = candidatos_carga_real[
        (candidatos_carga_real["origen"] == fila["zona_origen"]) & (candidatos_carga_real["destino"] == fila["zona_destino"])
    ]
    filas.append({
        "origen": fila["origen"], "destino": fila["destino"], "valorFlete": fila["valorFlete"],
        "n_candidatos_sap": len(candidatos),
        "sap_min": candidatos["valor"].min() if len(candidatos) else None,
        "sap_mediana": candidatos["valor"].median() if len(candidatos) else None,
        "sap_max": candidatos["valor"].max() if len(candidatos) else None,
    })

df_tarifario_ruta = pd.DataFrame(filas)
df_tarifario_ruta["diferencia_pct_vs_mediana"] = (
    (df_tarifario_ruta["valorFlete"] - df_tarifario_ruta["sap_mediana"]) / df_tarifario_ruta["sap_mediana"] * 100
)
df_tarifario_ruta["dentro_de_rango_sap"] = (
    (df_tarifario_ruta["valorFlete"] >= df_tarifario_ruta["sap_min"])
    & (df_tarifario_ruta["valorFlete"] <= df_tarifario_ruta["sap_max"])
)

print(f"Líneas Tarifario con match de ruta: {(df_tarifario_ruta['n_candidatos_sap'] > 0).sum()} de {len(df_tarifario_ruta)}")
print(f"Dentro del rango [min,max] de SAP para esa ruta: {df_tarifario_ruta['dentro_de_rango_sap'].sum()} de {len(df_tarifario_ruta)}")
print()
print(df_tarifario_ruta.to_string())
print()
print("Diferencia % vs. mediana SAP:")
print(df_tarifario_ruta["diferencia_pct_vs_mediana"].describe())

Líneas Tarifario con match de ruta: 32 de 32
Dentro del rango [min,max] de SAP para esa ruta: 32 de 32

          origen       destino  valorFlete  n_candidatos_sap    sap_min  sap_mediana    sap_max  diferencia_pct_vs_mediana  dentro_de_rango_sap
0   Buenaventura        Bogotá   6400000.0                58  1544000.0    4700000.0  7000000.0                  36.170213                 True
1           Cali        Bogotá   2300000.0                17  1650000.0    3361178.0  4915466.0                 -31.571610                 True
2         Bogotá     Cartagena   8650000.0                48  3000000.0    5600000.0  8840928.0                  54.464286                 True
3         Bogotá  Barranquilla   7667624.0                45  2600000.0    4880000.0  7900000.0                  57.123443                 True
4    Bucaramanga        Bogotá   2790000.0                13   900000.0    2500000.0  4064499.0                  11.600000                 True
5           Cali  Buenaventura  

## Paso 1c — Cruce real por ruta normalizada contra SICE-TAC

Mismo ejercicio para las 13 líneas `Fuente=Sicetac`, contra `piso-sicetac`
(núcleo + raras combinadas). Se excluye la línea con la ruta malformada de
prueba ("b"/"j") — buscar una sola letra como substring encuentra falsos
positivos en cualquier ciudad que la contenga.

Dos correcciones de método (revisión post-cierre), igual de importantes
que el cruce en sí: **(1)** `nombreruta` viene siempre como "ORIGEN -
DESTINO" — buscar cada ciudad como substring en cualquier parte del texto
(como se hacía antes) no distingue sentido, y una cotización Medellín→
Bogotá terminaba comparada contra la misma fila que Bogotá→Medellín, sin
verificar que la ruta real fuera en esa dirección. **(2)** `piso-sicetac`
incluye filas `condicioncarga="VACIO"` (tarifa de reposicionar un
vehículo vacío, no de transportar carga real) — no son comparables contra
un `ValorFlete` de una cotización con carga, y había una en el único
candidato encontrado antes. Se corrigen ambas cosas antes de comparar.

In [4]:
sicetac_todo = pd.concat([tablas["piso-sicetac_nucleo"], tablas["piso-sicetac_raras"]], ignore_index=True)

# (1) nombreruta = "ORIGEN - DESTINO" en el 100% de las filas (verificado) — partimos el
# texto para exigir que cada ciudad esté en su lado correcto, no en cualquier parte.
sicetac_todo[["origen_ruta_txt", "destino_ruta_txt"]] = sicetac_todo["nombreruta"].str.split(" - ", n=1, expand=True)

# (2) condicioncarga="VACIO" (nombretipocarga="Sin carga") es tarifa de vehículo vacío,
# no de carga real — se excluye antes de buscar candidatos comparables.
sicetac_carga_real = sicetac_todo[sicetac_todo["condicioncarga"] == "CARGADO"]

sicetac_ruta = base[(base["fuente"] == "Sicetac") & (base["origen_limpio"].str.len() > 2)].copy()

filas = []
for _, fila in sicetac_ruta.iterrows():
    candidatos = sicetac_carga_real[
        sicetac_carga_real["origen_ruta_txt"].str.contains(fila["origen_limpio"], na=False)
        & sicetac_carga_real["destino_ruta_txt"].str.contains(fila["destino_limpio"], na=False)
    ]
    filas.append({
        "origen": fila["origen"], "destino": fila["destino"], "valorFlete": fila["valorFlete"],
        "n_candidatos_sicetac": len(candidatos),
        "sicetac_valor": candidatos["valorsiceconhora"].median() if len(candidatos) else None,
    })

df_sicetac_ruta = pd.DataFrame(filas)
df_sicetac_ruta["diferencia_pct"] = (
    (df_sicetac_ruta["valorFlete"] - df_sicetac_ruta["sicetac_valor"]) / df_sicetac_ruta["sicetac_valor"] * 100
)

print(f"Líneas Sicetac válidas (excluida la malformada): {len(sicetac_ruta)}")
print(f"Con al menos 1 candidato SICE-TAC de carga real, misma dirección: {(df_sicetac_ruta['n_candidatos_sicetac'] > 0).sum()}")
print()
print(df_sicetac_ruta.to_string())

Líneas Sicetac válidas (excluida la malformada): 12
Con al menos 1 candidato SICE-TAC de carga real, misma dirección: 0

          origen       destino  valorFlete  n_candidatos_sicetac sicetac_valor diferencia_pct
0       Medellin        Bogota   4832788.0                     0          None            NaN
1       Medellin          Cali   1802013.0                     0          None            NaN
2         Bogota  Barranquilla   8175231.0                     0          None            NaN
3         Bogotá      Medellín   1564425.0                     0          None            NaN
4   Barranquilla        Bogotá   2818633.0                     0          None            NaN
5   Barranquilla        Bogotá   2818633.0                     0          None            NaN
6         Bogotá      Medellín   1564425.0                     0          None            NaN
7   Barranquilla        Bogotá   2818633.0                     0          None            NaN
8         Bogotá      Medellín   

## Paso 1d — Verificación de H-005: ¿se sostiene el +30,6% al angostar por vehículo/peso?

H-005 comparó las 32 líneas `Fuente=Tarifario` contra **todo el rango de
vehículo/peso** de tarifa-plaza para su ruta, porque no existía
traducción vehículo propio→SAP. Ya se consiguió esa traducción
(`tblSAPTiposVehiculoTarifario`, verificada 7/7 contra
`tarifa-plaza.tipovehiculo`). Este paso repite la comparación angostando
también por vehículo y rango de peso, reutilizando `tarifario_ruta`
(zona_origen/zona_destino, Paso 1b) y `candidatos_carga_real`
(lineanegocio≠"6", Paso 1b) tal cual.


In [5]:
catalogo_config_vehiculo = tablas["catalogo_configuraciones_vehiculo"]

tarifario_detalle = tarifario_ruta.merge(
    lineas_solicitud[["id", "configVehiculoId", "peso"]],
    left_on="lineaSolicitudId", right_on="id", how="left", suffixes=("", "_ls")
).merge(
    catalogo_config_vehiculo[["id", "tipoVehiculo"]],
    left_on="configVehiculoId", right_on="id", how="left", suffixes=("", "_cfg")
)

print(f"Líneas Tarifario: {len(tarifario_detalle)}")
print(f"Con ConfigVehiculoId poblado: {tarifario_detalle['configVehiculoId'].notna().sum()} de {len(tarifario_detalle)}")
print(f"Con TipoVehiculo (propio) resuelto vía ese Id: {tarifario_detalle['tipoVehiculo'].notna().sum()} de {len(tarifario_detalle)}")
print(f"Con Peso poblado: {tarifario_detalle['peso'].notna().sum()} de {len(tarifario_detalle)}")


Líneas Tarifario: 32
Con ConfigVehiculoId poblado: 0 de 32
Con TipoVehiculo (propio) resuelto vía ese Id: 0 de 32
Con Peso poblado: 32 de 32


**Bloqueo encontrado:** `ConfigVehiculoId` está NULL en el 100% de las 32
líneas `Fuente=Tarifario` (verificado también contra el CSV crudo de
`lineas_solicitud`, no es un artefacto de tipado del notebook 00, y
contra `tarifas_historicas`, que tampoco lo tiene poblado para estas
cotizaciones) — no hay ningún registro del vehículo cotizado en ninguna
tabla del set para estas líneas específicas. El diccionario
`tblSAPTiposVehiculoTarifario` en sí funciona (verificado 7/7 contra
`tarifa-plaza.tipovehiculo`), pero no hay nada que traducir: el filtro
de vehículo queda descartado para las 32 líneas, no por falla del
método sino por ausencia del dato de entrada. Sí se puede angostar por
**peso** (`peso` está poblado en las 32 líneas) — se compara a
continuación ruta+peso contra ruta sola, con fallback documentado para
las líneas que se queden sin ningún candidato dentro de su rango de
peso.


In [6]:
filas_peso = []
for _, fila in tarifario_detalle.iterrows():
    candidatos_ruta = candidatos_carga_real[
        (candidatos_carga_real["origen"] == fila["zona_origen"]) & (candidatos_carga_real["destino"] == fila["zona_destino"])
    ]
    candidatos_peso = candidatos_ruta[
        candidatos_ruta["rangopesoinf"].notna() & (candidatos_ruta["rangopesosup"] > 0)
        & (fila["peso"] >= candidatos_ruta["rangopesoinf"]) & (fila["peso"] <= candidatos_ruta["rangopesosup"])
    ]
    filas_peso.append({
        "origen": fila["origen"], "destino": fila["destino"], "peso": fila["peso"], "valorFlete": fila["valorFlete"],
        "n_candidatos_ruta": len(candidatos_ruta),
        "n_candidatos_peso": len(candidatos_peso),
        "sap_mediana_ruta": candidatos_ruta["valor"].median() if len(candidatos_ruta) else None,
        "sap_mediana_peso": candidatos_peso["valor"].median() if len(candidatos_peso) else None,
    })

df_peso = pd.DataFrame(filas_peso)

# Fallback explícito (no silencioso): si el rango de peso no deja NINGÚN candidato,
# usamos la mediana solo-ruta (Paso 1b) para esa línea en vez de descartarla.
df_peso["metodo"] = df_peso["n_candidatos_peso"].apply(lambda n: "ruta+peso" if n > 0 else "ruta (fallback, sin match de peso)")
df_peso["sap_mediana_final"] = df_peso["sap_mediana_peso"].fillna(df_peso["sap_mediana_ruta"])
df_peso["diferencia_pct_final"] = (df_peso["valorFlete"] - df_peso["sap_mediana_final"]) / df_peso["sap_mediana_final"] * 100

sin_candidato_peso = df_peso[df_peso["n_candidatos_peso"] == 0]
print(f"Líneas SIN ningún candidato al angostar por peso (usan fallback a ruta sola): {len(sin_candidato_peso)} de {len(df_peso)}")
print(sin_candidato_peso[["origen", "destino", "peso"]].drop_duplicates().to_string())
print()
print(df_peso[["origen", "destino", "peso", "valorFlete", "n_candidatos_ruta", "n_candidatos_peso", "metodo", "sap_mediana_final", "diferencia_pct_final"]].to_string())
print()
print("Diferencia % vs. mediana SAP — SOLO ruta (Paso 1b, referencia):")
print(df_tarifario_ruta["diferencia_pct_vs_mediana"].describe())
print()
print("Diferencia % vs. mediana SAP — ruta + peso (con fallback donde no hay match de peso):")
print(df_peso["diferencia_pct_final"].describe())
print()
print(f"Mediana ANTES (solo ruta): {df_tarifario_ruta['diferencia_pct_vs_mediana'].median():.1f}%")
print(f"Mediana DESPUÉS (ruta + peso): {df_peso['diferencia_pct_final'].median():.1f}%")


Líneas SIN ningún candidato al angostar por peso (usan fallback a ruta sola): 8 de 32
        origen destino     peso
4  Bucaramanga  Bogotá  21000.0
6       Bogotá  Cúcuta  15000.0

          origen       destino     peso  valorFlete  n_candidatos_ruta  n_candidatos_peso                              metodo  sap_mediana_final  diferencia_pct_final
0   Buenaventura        Bogotá  24000.0   6400000.0                 58                  4                           ruta+peso          6000000.0              6.666667
1           Cali        Bogotá   8500.0   2300000.0                 17                  1                           ruta+peso          3361178.0            -31.571610
2         Bogotá     Cartagena  28000.0   8650000.0                 48                  4                           ruta+peso          8225000.0              5.167173
3         Bogotá  Barranquilla  30000.0   7667624.0                 45                  2                           ruta+peso          7900000.0     

## Paso 1e — Reproducción exacta del algoritmo real (no una comparación contra la mediana)

Los Pasos 1b/1d comparan contra la **mediana** de SAP, pero eso no es lo
que hace CabinaFletes de verdad. En
`PropuestaTarifaAdapter.CandidatoSapAsync` (backend), la regla real es:
de las tarifas SAP que aplican a la ruta (y al vehículo/peso si hay
dato), **tomar la más alta** (`OrderByDescending(Valor).First()`) — no
un promedio ni la típica. Esto reencuadra todo H-005: comparar "lo más
alto que el sistema puede justificar" contra "la mediana del mercado" va
a dar positivo casi siempre, por diseño.

Aquí se reproduce esa regla real, con la mejor aproximación posible a
los datos que tenía el sistema **en el momento de cada cotización**
(usando `cotizacion_lineas.createdAt` como fecha de corte para
`vigencia`, no la fecha de hoy) + rango de peso + máximo. El vehículo
sigue sin poder aplicarse (Paso 1d). Si el valor reproducido coincide
EXACTO con `valorFlete`, confirma que el mecanismo es este y no un
sobreprecio manual.

In [7]:
filas_repro = []
for _, fila in tarifario_detalle.iterrows():
    fecha_cotizacion = fila["createdAt"]
    candidatos_vigentes = candidatos_carga_real[
        (candidatos_carga_real["origen"] == fila["zona_origen"]) & (candidatos_carga_real["destino"] == fila["zona_destino"])
        & (candidatos_carga_real["vigencia"] >= fecha_cotizacion)
    ]
    candidatos_peso = candidatos_vigentes[
        candidatos_vigentes["rangopesoinf"].notna() & (candidatos_vigentes["rangopesosup"] > 0)
        & (fila["peso"] >= candidatos_vigentes["rangopesoinf"]) & (fila["peso"] <= candidatos_vigentes["rangopesosup"])
    ]
    # misma regla del backend: si hay candidatos en el rango de peso, usar solo esos;
    # si no, usar el conjunto vigente más amplio (sin la restricción de peso)
    usar = candidatos_peso if len(candidatos_peso) > 0 else candidatos_vigentes
    top_reproducido = usar["valor"].max() if len(usar) > 0 else None
    filas_repro.append({
        "origen": fila["origen"], "destino": fila["destino"], "peso": fila["peso"],
        "fecha_cotizacion": fecha_cotizacion, "valorFlete": fila["valorFlete"],
        "n_vigentes": len(candidatos_vigentes), "n_vigentes_peso": len(candidatos_peso),
        "top_reproducido": top_reproducido,
    })

df_repro = pd.DataFrame(filas_repro)
df_repro["coincide_exacto"] = df_repro["top_reproducido"] == df_repro["valorFlete"]

print(f"Coinciden EXACTO con el algoritmo real (vigente a la fecha de cotización + mismo peso + la más alta): {df_repro['coincide_exacto'].sum()} de {len(df_repro)}")
print()
print(df_repro[["origen", "destino", "peso", "valorFlete", "n_vigentes", "n_vigentes_peso", "top_reproducido", "coincide_exacto"]].drop_duplicates(subset=["origen", "destino", "peso"]).to_string())


Coinciden EXACTO con el algoritmo real (vigente a la fecha de cotización + mismo peso + la más alta): 20 de 32

         origen       destino     peso  valorFlete  n_vigentes  n_vigentes_peso  top_reproducido  coincide_exacto
0  Buenaventura        Bogotá  24000.0   6400000.0          57                4        6000000.0            False
1          Cali        Bogotá   8500.0   2300000.0           8                0        4800000.0            False
2        Bogotá     Cartagena  28000.0   8650000.0           2                1        8650000.0             True
3        Bogotá  Barranquilla  30000.0   7667624.0           2                0        7667624.0             True
4   Bucaramanga        Bogotá  21000.0   2790000.0           2                0        2220000.0            False
5          Cali  Buenaventura   9000.0   2200000.0           4                0        2200000.0             True
6        Bogotá        Cúcuta  15000.0   6000000.0           2                0        600

**Nota sobre las líneas que no coinciden exacto:** se agrupan en solo 3
rutas únicas (de 8), cada una repetida ~4 veces en las 32 líneas — no es
ruido disperso. Se probó además sumar `historico-sap` como fuente
adicional (el motor real también lo consulta) deduplicando por
`idTarifa` a la versión más reciente del log; el resultado **empeoró**
(12/32 en vez de 20/32), así que se descartó — no es la pieza que falta.
La causa más probable de las 3 rutas restantes es una limitación ya
documentada: no existe una foto histórica de qué `vigencia` tenía cada
tarifa exactamente el día de cada cotización (`vigencia` es un campo que
se puede editar después de creado; solo vemos su valor actual — ver
`limitaciones.md` L-02), sumado a la falta de vehículo (L-13). No se
persigue el 100% — el rastro se acaba en datos que este extracto no
tiene.

## Paso 1f — ¿Sirve `tblRangoPeso` para inferir el vehículo cuando `ConfigVehiculoId` falta? (L-13)

`tblSAPTiposVehiculoTarifario` traduce vehículo propio→SAP, pero L-13
sigue bloqueando su uso: no hay vehículo registrado en ninguna de las 32
líneas. `tblRangoPeso` (rango de peso oficial por tipo de vehículo
propio) ofrece una vía indirecta: inferir el vehículo a partir del
`peso` de la línea. Se prueba aquí si esa inferencia aporta algo a la
comparación de H-005.

In [8]:
rango_peso = pd.read_csv("../../00-fuentes/fuentes-corporativas/tblRangoPeso_202609142053.csv")
tipos_vehiculo_tarifario = pd.read_csv("../../00-fuentes/fuentes-corporativas/tblSAPTiposVehiculoTarifario_202609142052.csv")
dict_vehiculo_sap = (
    tipos_vehiculo_tarifario.assign(codigo_norm=tipos_vehiculo_tarifario["CODIGO"].str.strip().str.upper())
    .groupby("codigo_norm")["ID"].first().astype(str).to_dict()
)


def inferir_vehiculos_por_peso(peso):
    cand = rango_peso[(peso >= rango_peso["PESOMINIMO"]) & (peso <= rango_peso["PESOMAXIMO"])]
    return sorted(set(cand["TIPOVEHICULO"]))


filas_infer = []
for _, fila in tarifario_detalle.drop_duplicates(subset=["zona_origen", "zona_destino", "peso"]).iterrows():
    vehiculos_propios = inferir_vehiculos_por_peso(fila["peso"])
    ids_sap_inferidos = sorted({dict_vehiculo_sap.get(v.strip().upper()) for v in vehiculos_propios} - {None})

    candidatos_ruta_peso = candidatos_carga_real[
        (candidatos_carga_real["origen"] == fila["zona_origen"]) & (candidatos_carga_real["destino"] == fila["zona_destino"])
        & candidatos_carga_real["rangopesoinf"].notna() & (candidatos_carga_real["rangopesosup"] > 0)
        & (fila["peso"] >= candidatos_carga_real["rangopesoinf"]) & (fila["peso"] <= candidatos_carga_real["rangopesosup"])
    ]
    filas_infer.append({
        "origen": fila["origen"], "destino": fila["destino"], "peso": fila["peso"],
        "vehiculos_propios_inferidos": vehiculos_propios, "ids_sap_inferidos": ids_sap_inferidos,
        "n_candidatos_ruta_peso": len(candidatos_ruta_peso),
        "con_tipovehiculo_poblado": candidatos_ruta_peso["tipovehiculo"].notna().sum(),
    })

print(pd.DataFrame(filas_infer).to_string())


         origen       destino     peso vehiculos_propios_inferidos ids_sap_inferidos  n_candidatos_ruta_peso  con_tipovehiculo_poblado
0  Buenaventura        Bogotá  24000.0                       [3S3]               [1]                       4                         0
1          Cali        Bogotá   8500.0                    [2, 2S2]           [11, 2]                       1                         1
2        Bogotá     Cartagena  28000.0                       [3S3]               [1]                       4                         0
3        Bogotá  Barranquilla  30000.0                       [3S3]               [1]                       2                         0
4   Bucaramanga        Bogotá  21000.0                  [2S3, 3S3]            [1, 5]                       0                         0
5          Cali  Buenaventura   9000.0                       [2S2]               [2]                       2                         0
6        Bogotá        Cúcuta  15000.0                 

**Conclusión: no ayuda, y por la misma razón ya documentada en el Paso
1e.** `tblRangoPeso` sí infiere un candidato de vehículo por peso (único
en 5 de 8 rutas: 24000→3S3, 28000→3S3, 30000→3S3, 21000/22000 dan
2 candidatos, 15000/8500 también dan 2). Pero en 7 de las 8 rutas, los
candidatos de `tarifa-plaza` ya angostados por ruta+peso tienen
`tipovehiculo` **vacío en el 100% de las filas** — no hay nada contra
qué comparar el vehículo inferido. Confirma otra vez que estas rutas
(Puerto a Interior/Interior a Puerto) tarifican por peso, no por
vehículo — el mismo hallazgo del bloqueo de `ConfigVehiculoId` (Paso
1d), visto ahora desde otro ángulo. No se usa esta inferencia en H-005.

## Paso 1g — SICE-TAC con el extracto real completo de agosto 2026 (no la muestra de 151 filas)

El Paso 1c concluyó "0 de 12 líneas con candidato comparable" porque
`piso-sicetac` es una muestra curada de 151 filas que no fue diseñada
para cubrir estas rutas. Ya se consiguió el extracto **real completo**
de SICE-TAC para el período 202608 (agosto 2026, el vigente para estas
12 cotizaciones) — el mismo cubo `WF_CUBO1_26S` que usa el sistema real,
~2,1 millones de filas
(`analitica/00-fuentes/Historicos_SICE/SICETAC_20260801.xlsx`).

Este período viene repartido en 3 hojas de Excel (el límite de ~1M
filas por hoja). La celda siguiente lee las 3 hojas completas y se
queda solo con las filas de las 4 ciudades involucradas (Medellín,
Bogotá, Cali, Barranquilla) con `CONDICIONCARGA="CARGADO"` — **tarda
~6-7 minutos**, es la única forma de leer el archivo completo (los
metadatos de dimensión del archivo están mal escritos, así que no se
puede pedirle a Excel/pandas que lea "solo un rango": hay que recorrer
las ~2,1 millones de filas igual).

La columna de valor final (`VALORSICECONHORA` en `piso-sicetac`) no
viene calculada en el extracto crudo — se reconstruye con la fórmula
verificada 151/151 contra la muestra ya conocida:
`valorsiceconhora = VALORMOVILIZA + VALORHORA × horas_pactadas`, donde
`horas_pactadas` es una constante por tipo de carga (8 horas para todas
las categorías, 2 horas solo para "Contenedor vacío" — verificado
también 151/151, sin excepciones, contra la muestra).

In [9]:
import openpyxl

RUTA_SICETAC_AGOSTO = "../../00-fuentes/Historicos_SICE/SICETAC_20260801.xlsx"
CIUDADES_INVOLUCRADAS = ["MEDELLIN", "BOGOTA", "CALI", "BARRANQUILLA"]
COLUMNAS_SICETAC = [
    "PERIODO", "FECHAINGRESO", "ORIGEN", "NOMORIGEN", "DESTINO", "NOMDESTINO",
    "CONFIGURACION", "TIPOCARGA", "NOMBRETIPOCARGA", "UNIDADTRANSPORTE",
    "NOMBREUNIDADTRANSPORTE", "CONDICIONCARGA", "KILOMETROS", "VALORMOVILIZA",
    "VALORHORA", "HORASRECORRIDO", "VIAESTANDAR", "RUTASID", "VIA",
]

wb = openpyxl.load_workbook(RUTA_SICETAC_AGOSTO, read_only=True, data_only=True)
filas_candidatas = []
for nombre_hoja in ["Exportar Hoja de Trabajo", "Sheet1", "Sheet2"]:
    ws = wb[nombre_hoja]
    # min_row=2 salta el encabezado (hoja 1) o la fila en blanco (hojas 2-3);
    # max_row explícito porque los metadatos de dimensión del archivo están mal
    # escritos (dicen max_row=1) y openpyxl detendría la lectura de inmediato.
    for fila in ws.iter_rows(min_row=2, max_row=3_000_000, max_col=19, values_only=True):
        if fila[3] is None or fila[5] is None or fila[11] != "CARGADO":
            continue
        if any(c in fila[3] for c in CIUDADES_INVOLUCRADAS) and any(c in fila[5] for c in CIUDADES_INVOLUCRADAS):
            filas_candidatas.append(fila)
wb.close()

sicetac_agosto = pd.DataFrame(filas_candidatas, columns=COLUMNAS_SICETAC)
horas_pactadas = sicetac_agosto["NOMBRETIPOCARGA"].apply(lambda t: 2.0 if t == "Contenedor vacío" else 8.0)
sicetac_agosto["valorsiceconhora"] = sicetac_agosto["VALORMOVILIZA"] + sicetac_agosto["VALORHORA"] * horas_pactadas
sicetac_agosto["prioridad_via"] = (sicetac_agosto["VIAESTANDAR"] != "SI").astype(int)

print(f"Filas candidatas extraídas (4 ciudades, CARGADO): {len(sicetac_agosto)}")


C:\Users\Scalderon\Documents\Proyectos Botero Soto\Nexo\NEXO\analitica\03-eda-extraccion-4-fuentes\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Filas candidatas extraídas (4 ciudades, CARGADO): 4050


Con las candidatas ya extraídas, se reproduce `SeleccionarSice`
(backend, `PropuestaTarifaAdapter.cs`): si hay vehículo (`ConfiguracionVehiculo.TipoVehiculo`),
angostar por `CONFIGURACION`; sin `tipologia` (null en el 100% de estas
12 líneas), angostar a `NOMBRETIPOCARGA="General"`; si hay `carroceria`,
angostar por `NOMBREUNIDADTRANSPORTE`; y del resultado, tomar primero
`VIAESTANDAR="SI"`, luego menos `HORASRECORRIDO`, luego el `VALORMOVILIZA`
más bajo.

In [10]:
sicetac_ruta_detalle = sicetac_ruta.merge(
    lineas_solicitud[["id", "configVehiculoId"]], left_on="lineaSolicitudId", right_on="id", how="left", suffixes=("", "_ls")
).merge(
    catalogo_config_vehiculo[["id", "tipoVehiculo", "carroceria"]], left_on="configVehiculoId", right_on="id", how="left", suffixes=("", "_cfg")
)

filas_final = []
for _, fila in sicetac_ruta_detalle.iterrows():
    origen, destino = fila["origen_limpio"], fila["destino_limpio"]
    cand = sicetac_agosto[
        sicetac_agosto["NOMORIGEN"].str.contains(origen, na=False) & sicetac_agosto["NOMDESTINO"].str.contains(destino, na=False)
    ]

    vehiculo = fila["tipoVehiculo"]
    if pd.notna(vehiculo):
        v = vehiculo.strip().upper()
        cand_config = cand[cand["CONFIGURACION"].str.strip().str.upper() == v]
        if len(cand_config) == 0:
            cand_config = cand[cand["CONFIGURACION"].str.strip().str.upper().str.startswith(v)]
        if len(cand_config) > 0:
            cand = cand_config

    cand_general = cand[cand["NOMBRETIPOCARGA"] == "General"]
    cand = cand_general if len(cand_general) > 0 else cand

    carroceria = fila["carroceria"]
    if pd.notna(carroceria):
        cand_carro = cand[cand["NOMBREUNIDADTRANSPORTE"].str.strip().str.upper() == carroceria.strip().upper()]
        if len(cand_carro) > 0:
            cand = cand_carro

    top = cand.sort_values(["prioridad_via", "HORASRECORRIDO", "VALORMOVILIZA"]).iloc[0] if len(cand) > 0 else None
    filas_final.append({
        "origen": fila["origen"], "destino": fila["destino"], "valorFlete": fila["valorFlete"],
        "vehiculo": vehiculo, "carroceria": carroceria, "n_candidatos_final": len(cand),
        "sicetac_reproducido": top["valorsiceconhora"] if top is not None else None,
    })

df_sicetac_real = pd.DataFrame(filas_final)
df_sicetac_real["coincide_exacto"] = df_sicetac_real["sicetac_reproducido"] == df_sicetac_real["valorFlete"]

print(f"Coinciden EXACTO con el algoritmo real de SICE-TAC: {df_sicetac_real['coincide_exacto'].sum()} de {len(df_sicetac_real)}")
print()
print(df_sicetac_real.to_string())


Coinciden EXACTO con el algoritmo real de SICE-TAC: 12 de 12

          origen       destino  valorFlete vehiculo  carroceria  n_candidatos_final  sicetac_reproducido  coincide_exacto
0       Medellin        Bogota   4832788.0      3S3      FURGON                   2            4832788.0             True
1       Medellin          Cali   1802013.0      2L2     ESTACAS                   3            1802013.0             True
2         Bogota  Barranquilla   8175231.0      3S3  Plataforma                   2            8175231.0             True
3         Bogotá      Medellín   1564425.0      NaN         NaN                  96            1564425.0             True
4   Barranquilla        Bogotá   2818633.0      NaN         NaN                  96            2818633.0             True
5   Barranquilla        Bogotá   2818633.0      NaN         NaN                  96            2818633.0             True
6         Bogotá      Medellín   1564425.0      NaN         NaN                  96 

## Paso 1h — Bogotá tiene 2 zonas SAP válidas: revisando las 12 líneas SAP que no coincidían (Paso 1e)

Pregunta que surgió revisando el Paso 1e con el usuario: si H-007 ya
confirmó que las 32 líneas Tarifario tienen su `ValorFlete` exacto en
algún lado de `tarifa-plaza`, ¿por qué el Paso 1e no lo encontraba para
12 de ellas? Se buscó el valor exacto de esas 3 rutas sin restringir
zona — y apareció bajo `ZTC_NORTE_BOGOTA`, una **segunda zona SAP para
Bogotá** (`tarifa-plaza` tiene exactamente 2: `ZTC_BOGOTA` y
`ZTC_NORTE_BOGOTA`) que ni rapidfuzz ni las tablas reales de ciudad→zona
(H-006) podían descubrir — ambos métodos resuelven por nombre de
ciudad, no distinguen que una misma ciudad pueda tener más de una zona
según la ruta.

Se probó cada zona por separado (no una unión de candidatos — mezclar
ambas zonas da un resultado peor, ver más abajo) para las 3 rutas que no
coincidían.


In [11]:
def top_para_zona(origen_zona, destino_zona, peso, fecha_cotizacion):
    cand = candidatos_carga_real[
        (candidatos_carga_real["origen"] == origen_zona) & (candidatos_carga_real["destino"] == destino_zona)
        & (candidatos_carga_real["vigencia"] >= fecha_cotizacion)
    ]
    cand_peso = cand[
        cand["rangopesoinf"].notna() & (cand["rangopesosup"] > 0)
        & (peso >= cand["rangopesoinf"]) & (peso <= cand["rangopesosup"])
    ]
    usar = cand_peso if len(cand_peso) > 0 else cand
    return usar["valor"].max() if len(usar) > 0 else None


filas_h = []
no_coincidian = df_repro[~df_repro["coincide_exacto"]].drop_duplicates(subset=["origen", "destino", "peso"])
for _, fila_orig in tarifario_detalle.iterrows():
    clave = (fila_orig["origen"], fila_orig["destino"], fila_orig["peso"])
    if clave not in set(map(tuple, no_coincidian[["origen", "destino", "peso"]].values)):
        continue
    fecha = fila_orig["createdAt"]
    zo, zd = fila_orig["zona_origen"], fila_orig["zona_destino"]

    variantes = {"zona original": (zo, zd)}
    if zo == "ZTC_BOGOTA":
        variantes["origen -> Bogotá Norte"] = ("ZTC_NORTE_BOGOTA", zd)
    if zd == "ZTC_BOGOTA":
        variantes["destino -> Bogotá Norte"] = (zo, "ZTC_NORTE_BOGOTA")

    for etiqueta, (zo2, zd2) in variantes.items():
        top = top_para_zona(zo2, zd2, fila_orig["peso"], fecha)
        filas_h.append({
            "origen": fila_orig["origen"], "destino": fila_orig["destino"], "peso": fila_orig["peso"],
            "valorFlete": fila_orig["valorFlete"], "variante": etiqueta,
            "top_reproducido": top, "coincide_exacto": top == fila_orig["valorFlete"],
        })

df_h = pd.DataFrame(filas_h).drop_duplicates(subset=["origen", "destino", "peso", "variante"])
print(df_h.to_string())


         origen destino     peso  valorFlete                 variante  top_reproducido  coincide_exacto
0  Buenaventura  Bogotá  24000.0   6400000.0            zona original        6000000.0            False
1  Buenaventura  Bogotá  24000.0   6400000.0  destino -> Bogotá Norte        6400000.0             True
2          Cali  Bogotá   8500.0   2300000.0            zona original        4800000.0            False
3          Cali  Bogotá   8500.0   2300000.0  destino -> Bogotá Norte        2300000.0             True
4   Bucaramanga  Bogotá  21000.0   2790000.0            zona original        2220000.0            False
5   Bucaramanga  Bogotá  21000.0   2790000.0  destino -> Bogotá Norte              NaN            False


In [12]:
zonas_relacion = pd.read_csv("../../00-fuentes/fuentes-corporativas/tblZonasTarificacionRelacion_202609142051.csv")
bogota_relacion = zonas_relacion[zonas_relacion["IDZONATARIFICACION"].isin(["ZTC_BOGOTA", "ZTC_NORTE_BOGOTA"])]
print("Municipios satelite oficiales de cada zona de Bogota:")
print(bogota_relacion[["IDZONATARIFICACION", "IDZONATRANSPORTE"]].to_string())

# Bogota como ciudad esta mapeada a ZTC_BOGOTA, no a NORTE:
print()
print("¿'ZC_BOGOTA' (la ciudad) esta en la lista?:", "ZC_BOGOTA" in bogota_relacion["IDZONATRANSPORTE"].values)

# Comparar el par de tablas de precio para Buenaventura -> ambas zonas de Bogota,
# mismo tipo de vehiculo, para ver si son 2 tablas paralelas (no un duplicado casual)
par = tarifa_plaza[
    (tarifa_plaza["origen"] == "ZC_BUENAVENTURA") & (tarifa_plaza["destino"].isin(["ZTC_BOGOTA", "ZTC_NORTE_BOGOTA"]))
    & (tarifa_plaza["rangopesoinf"] == 0) & (tarifa_plaza["lineanegocio"] == "1")
].drop_duplicates(subset=["destino", "tipovehiculo"])
print()
print("Comparación de precio por vehículo, Buenaventura -> cada zona de Bogotá (tarifa 'cualquier peso'):")
print(par[["destino", "tipovehiculo", "valor"]].sort_values(["tipovehiculo", "destino"]).to_string())


Municipios satelite oficiales de cada zona de Bogota:
   IDZONATARIFICACION    IDZONATRANSPORTE
77         ZTC_BOGOTA           ZC_BOGOTA
78         ZTC_BOGOTA            ZC_FUNZA
79         ZTC_BOGOTA           ZC_MADRID
80         ZTC_BOGOTA   ZC_MOSQUERA (CUN)
81         ZTC_BOGOTA     ZC_SOACHA (CUN)
82         ZTC_BOGOTA             ZC_COTA
83         ZTC_BOGOTA            ZC_TENJO
84         ZTC_BOGOTA       ZC_FACATATIVA
85         ZTC_BOGOTA         ZC_EL ROSAL
86         ZTC_BOGOTA           ZC_SIBATE
87   ZTC_NORTE_BOGOTA        ZC_TOCANCIPA
88   ZTC_NORTE_BOGOTA             ZC_CHIA
89   ZTC_NORTE_BOGOTA           ZC_CAJICA
90   ZTC_NORTE_BOGOTA            ZC_TABIO
91   ZTC_NORTE_BOGOTA       ZC_GACHANCIPA
92   ZTC_NORTE_BOGOTA             ZC_SOPO
93   ZTC_NORTE_BOGOTA  ZC_ZIPAQUIRA (CUN)
94   ZTC_NORTE_BOGOTA          ZC_NEMOCON
95   ZTC_NORTE_BOGOTA            ZC_COGUA

¿'ZC_BOGOTA' (la ciudad) esta en la lista?: True

Comparación de precio por vehículo, Buenaventura -> cad

**Resultado: 2 de las 3 rutas calzan con la zona "Bogotá Norte" — pero
la explicación no es una regla de corredor de origen, es más probable
que el destino real fuera más específico que "Bogotá".**

`tblZonasTarificacionRelacion` deja claro que la ciudad de Bogotá en sí
está mapeada oficialmente a `ZTC_BOGOTA` (junto con satélites al
occidente/sur: Funza, Madrid, Mosquera, Soacha, Cota, Tenjo,
Facatativá, El Rosal, Sibaté) — `ZTC_NORTE_BOGOTA` es la zona de los
satélites al **norte** (Chía, Cajicá, Zipaquirá, Tocancipá, Sopó,
Tabio, Gachancipá, Nemocón, Cogua). Revisando `tarifa-plaza` completa
para estos orígenes, **son 2 tablas de precio paralelas**: para casi
cada combinación de peso/vehículo hay un par de filas, una por zona, y
la zona Norte cobra sistemáticamente más (5-10% más) — no es un
duplicado ni un error, es una diferenciación de precio real por
destino más fino dentro del área metropolitana.

Dado que la ciudad de Bogotá por sí sola no debería mapear a la zona
Norte, la explicación más probable de que el valor exacto de
`Buenaventura→Bogotá` y `Cali→Bogotá` (8 de las 12 líneas) aparezca ahí
es que el destino real de esas 2 solicitudes de prueba era un municipio
satélite del norte (ej. Chía o Zipaquirá) — y quedó simplificado a
"Bogotá" en `catalogo_rutas`. **No se puede confirmar con certeza** cuál
era el destino exacto con los datos disponibles; se documenta como
explicación plausible, no confirmada. `Bucaramanga→Bogotá` (4 líneas)
sigue sin candidato exacto en ninguna de las 2 zonas.

**Resultado numérico de H-005 (SAP): 28 de 32 líneas (87,5%) coinciden
exacto** — sube desde el 62,5% del Paso 1e. Solo `Bucaramanga→Bogotá`
(4 líneas) queda sin explicar. La lección de fondo no cambia: **una
ciudad puede tener más de una zona SAP válida según qué tan específico
sea el destino real**, y eso es indetectable tanto por fuzzy matching
como por el catálogo real de ciudad→zona (H-006) cuando el propio dato
de destino (`catalogo_rutas.destino`) es más genérico que la zona real
de entrega.

## Paso 1i — Cerrando Bucaramanga→Bogotá: valor real, pero de una tarifa de Devolución Contenedor

Las 4 líneas de `Bucaramanga→Bogotá` (peso 21.000 kg, `ValorFlete`
$2.790.000) seguían sin candidato exacto en ninguna de las 2 zonas de
Bogotá ni en ninguna fecha de vigencia. Se buscó el valor exacto sin
restringir ruta ni `lineanegocio` — salen 7 filas en `tarifa-plaza` con
ese mismo valor, la mayoría coincidencias de otras rutas sin relación
(`ZTC_CARTAGENA→ZC_GUATAVITA`, `ZC_BUENAVENTURA→ZTC_ESPINAL_TOL`). De
esas 7, una sola corresponde a la ruta real: `ZTC_BUCARAMANGA→ZTC_BOGOTA`,
bajo `lineanegocio="6"` (Devolución Contenedor, la categoría que se
excluye a propósito de todas las demás comparaciones de H-005 por ser
reposicionamiento de contenedor vacío, no carga real).

In [13]:
fila_exacta = tarifa_plaza[tarifa_plaza["valor"] == 2790000.0]
print(f"Filas con valor exacto $2.790.000 en tarifa-plaza (sin restringir ruta): {len(fila_exacta)}")
print(fila_exacta[[
    "id", "origen", "destino", "valor", "vigencia", "rangopesoinf", "rangopesosup",
    "lineanegocio", "lineaNegocioDescripcion", "exclusivo", "detallestarifa",
]].to_string())

print("\nDe esas, la única para la ruta real Bucaramanga->Bogota:")
fila_relevante = fila_exacta[
    (fila_exacta["origen"] == "ZTC_BUCARAMANGA") & (fila_exacta["destino"] == "ZTC_BOGOTA")
]
print(fila_relevante[[
    "id", "origen", "destino", "valor", "vigencia", "rangopesoinf", "rangopesosup",
    "lineanegocio", "lineaNegocioDescripcion", "exclusivo", "detallestarifa",
]].to_string())


Filas con valor exacto $2.790.000 en tarifa-plaza (sin restringir ruta): 7
          id           origen          destino      valor   vigencia  rangopesoinf  rangopesosup lineanegocio                     lineaNegocioDescripcion  exclusivo                      detallestarifa
2861   12681  ZTC_BUCARAMANGA       ZTC_BOGOTA  2790000.0 2026-09-10        4000.0        4000.0            6  Oferta de Expedición Devolución Contenedor       True  Flete Libre de Cargue y Descargue.
11759  21579  ZC_BUENAVENTURA  ZTC_ESPINAL_TOL  2790000.0 2025-10-10           0.0           0.0            1      Oferta de Expedición Puerto a interior      False                                 NaN
18140  28002    ZTC_CARTAGENA     ZC_GUATAVITA  2790000.0 2025-11-10           0.0           0.0            4               Oferta de Expedición Nacional      False                                 NaN
18141  28003    ZTC_CARTAGENA     ZC_GUATAVITA  2790000.0 2025-11-10           0.0           0.0            4            

La fila (`ZTC_BUCARAMANGA → ZTC_BOGOTA`, $2.790.000, rango de peso
4.000-4.000 kg — el peso típico de un contenedor de 40 pies vacío,
`exclusivo=True`, `"Flete Libre de Cargue y Descargue"`) es consistente
con una tarifa real de devolución de contenedor. Pero en
`lineas_solicitud`, estas 3 líneas de solicitud tienen
`RetornaContenedor=False` y `peso=21000` — el contexto de la solicitud
propia **no** coincide con el de la tarifa encontrada.

**Confirmado por el usuario:** el valor de estas 4 líneas se tomó
manualmente de un Excel de referencia al armar la prueba, no lo generó
el motor de propuestas en vivo — por eso el número es una tarifa SAP
real y verificable, pero su contexto (peso, línea de negocio) no
coincide con el de la cotización de prueba a la que se le asignó. No es
un caso más para el algoritmo de reproducción (Pasos 1e/1h): es captura
manual, igual en espíritu a `Fuente=Manual` (H-002), aunque aquí sí es
un valor real de mercado, no inventado.

**Cierre de H-005 (lado SAP):** de las 32 líneas `Fuente=Tarifario`,
28 (87,5%) se explican por el algoritmo real (Pasos 1e/1h) y las 4
restantes (Bucaramanga→Bogotá) por captura manual desde Excel,
confirmado directamente por el usuario — las 32 quedan con una
explicación, aunque no las 32 por el mismo mecanismo.


## Paso 2 — Dispersión de VALOR en SAP, por LineaNegocio (§7.3 del plan)

Sin depender del cruce con lo propio: ¿qué tan distinto es el precio de
una tarifa SAP según la categoría de negocio? Esto caracteriza el
mercado por sí mismo, información con valor propio para el futuro
modelo, independiente de si se puede cruzar contra CabinaFletes o no.


In [14]:
tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]

por_linea_negocio = tarifa_plaza.groupby("lineaNegocioDescripcion")["valor"].describe()
print(por_linea_negocio.to_string())



                                              count          mean           std        min        25%        50%        75%         max
lineaNegocioDescripcion                                                                                                                
Oferta de Expedición Devolución Contenedor   1028.0  1.863085e+06  1.439189e+06        0.0   900000.0  1300001.0  2500000.0   8694464.0
Oferta de Expedición Interior a Puerto       2340.0  4.041846e+06  2.270570e+06   220000.0  2232500.0  3600000.0  5500000.0  13272000.0
Oferta de Expedición Licitaciones               1.0  3.556418e+06           NaN  3556418.0  3556418.0  3556418.0  3556418.0   3556418.0
Oferta de Expedición Nacional               13625.0  3.163931e+06  2.038501e+06   100000.0  1600000.0  2700000.0  4200000.0  18000000.0
Oferta de Expedición Puerto a interior       6023.0  4.030362e+06  2.411012e+06   100000.0  2000000.0  3600000.0  5600000.0  15700000.0


## Paso 3 — Dispersión de VALOR en SAP, por tipo de vehículo (§7.3 del plan)

Mismo ejercicio que LineaNegocio, ahora por TIPOVEHICULO (7 categorías,
ya confirmadas 1 a 1 con su descripción en el notebook 04). Misma
salvedad: mezcla variación real de vehículo con variación de ruta.


In [15]:
por_tipo_vehiculo = tarifa_plaza.groupby("vehiculoDescripcion")["valor"].describe()
print(por_tipo_vehiculo.to_string())


                                           count          mean           std       min        25%        50%        75%         max
vehiculoDescripcion                                                                                                                
Dobletroque                                438.0  3.641868e+06  1.390625e+06  700000.0  2800000.0  3500000.0  4600000.0   8000000.0
Liviano (capacidad 4 ton)                 5221.0  1.826780e+06  9.436562e+05  220000.0  1100000.0  1600000.0  2500000.0   7500000.0
Liviano mas 10.5 ton (capacidad 8.5 ton)  5147.0  2.377126e+06  1.191969e+06  100000.0  1400000.0  2200000.0  3224000.0   8500000.0
Mini Mula Poderosa                         421.0  4.686166e+06  1.780740e+06  700000.0  3500000.0  4760000.0  5800000.0   8800000.0
Minimula                                  3488.0  4.087465e+06  1.886331e+06  570000.0  2700000.0  4000000.0  5200000.0  12500000.0
Rígidos, Turbo o Sencillo                   14.0  2.315387e+06  1.371188e+06

## Paso 4 — Dispersión de VALORSICECONHORA en SICE-TAC, por tipo de carga (§7.3 del plan)

Solo sobre `piso-sicetac_nucleo` (126 filas) — las 25 `raras` se dejan
fuera a propósito, son casos extremos elegidos adrede, mezclarlas
sesgaría el resultado hacia arriba/abajo artificialmente.


In [16]:
piso_sicetac_nucleo = tablas["piso-sicetac_nucleo"]
por_tipo_carga = piso_sicetac_nucleo.groupby("nombretipocarga")["valorsiceconhora"].describe()
print(por_tipo_carga.to_string())


                    count          mean           std        min         25%        50%         75%        max
nombretipocarga                                                                                               
Carga Refrigerada     6.0  2.648267e+06  1.502266e+06   913366.0  1434634.25  2619199.5  3617280.25  4748086.0
Contenedor cargado    6.0  4.113858e+06  2.719678e+06  1474516.0  2141309.75  3294614.5  5678825.25  8388967.0
Contenedor vacío      6.0  2.042604e+06  1.682862e+06   330531.0   930772.50  1921094.0  2300937.25  5054689.0
General              24.0  2.568956e+06  1.383970e+06   705009.0  1484076.75  2224079.5  3426629.00  5406168.0
Granel Sólido        30.0  3.050617e+06  1.646392e+06   367451.0  1743835.00  3197555.0  4459708.25  6251385.0
Granel líquido        6.0  4.115099e+06  3.157665e+06  1017941.0  1952659.25  3267965.0  5408030.75  9501111.0
Sin carga            48.0  1.668203e+06  1.366734e+06    39366.0   549494.25  1258556.5  2307198.00  6060042.0


## Paso 5 — Fuente × Resultado de la cotización (§7.4 del plan)

Modalidad × líneas de negocio SAP no se puede — Modalidad tiene solo 1
valor real en las 13 solicitudes. Este otro cruce sí es posible:
¿la fuente que usó CabinaFletes para cotizar tiene alguna relación con
si la cotización se ganó o no? Con 18 cotizaciones, el plan mismo avisa
que cualquier patrón aquí es anecdótico, no estadístico — se reporta
igual, con esa salvedad explícita.


In [17]:
cotizaciones = tablas["cotizaciones"]
cl_con_resultado = cotizacion_lineas.merge(
    cotizaciones[["id", "resultado"]], left_on="cotizacionId", right_on="id", suffixes=("", "_cot")
)
tabla_cruzada = pd.crosstab(cl_con_resultado["fuente"], cl_con_resultado["resultado"])
print(tabla_cruzada)


resultado  Ganada  Perdida  SinDefinir
fuente                                
Manual          2        0           1
Sicetac         2        2           9
Tarifario       8        8          16


## Síntesis del notebook 05 — Bivariado y multivariado (Fase 6, §7 del plan)

**Objetivo del plan:** responder qué tan cerca cotiza CabinaFletes del
piso/plaza de mercado — el corazón de esta HU. Resultado: **respondido
para SAP y para SICE-TAC, las 32+12 líneas quedan explicadas** — no es
que CabinaFletes cotice "por encima" o "por debajo" del mercado, es que
el sistema aplica una regla de selección determinística sobre las
tarifas oficiales (confirmada exacta en 28/32 SAP + 12/12 SICE-TAC), y
las 4 líneas SAP restantes resultaron ser captura manual desde Excel,
confirmada por el usuario.

**Primer intento, por código de negocio (§7.1/§7.2, Paso 1): 0 casos
utilizables.** Solo 5 líneas de solicitud tienen `LineaNegocio`/
`ClaseViaje` poblados; de esas, solo 3 llegaron a cotizarse — y las 3
resultan ser casos ya conocidos como ruido (la ruta malformada
`"b"/"j"`, y los 2 valores `Fuente=Manual` ya descartados por absurdos).
Comparar solo por código de negocio (sin ruta) tampoco habría sido
válido de todas formas — la misma combinación de `LineaNegocio`/
`ClaseViaje` cubre miles de rutas SAP con precios muy distintos entre sí.

**Segundo intento, por ruta normalizada (Pasos 1b-1c), comparando contra
la mediana/muestra curada.** Una vez resuelta la ruta con `rapidfuzz`
(notebook 06, Fase 8):
- **Contra SAP (Tarifario, 32 de 32 líneas):** las 32 caen dentro del
  rango [mínimo, máximo] de SAP para su ruta, con mediana de diferencia
  +30,6% sobre la mediana SAP. **Salvedad inicial:** comparación contra
  todo el rango de vehículo/peso de la ruta, no vehículo-por-vehículo.
- **Contra SICE-TAC (Sicetac): 0 de 12 líneas con candidato — pero por
  el diseño de la muestra, no por falla del sistema.** `piso-sicetac`
  (151 filas) es una muestra curada que no fue diseñada para cubrir
  Medellín/Bogotá/Cali/Barranquilla — no había manera de que estas 12
  líneas encontraran candidato ahí.

**Verificación post-cierre, en cinco pasos (Pasos 1d-1e-1g-1h-1i):
ninguno de los "gaps" originales era lo que parecía.**

*Paso 1d — angostar SAP por vehículo/peso:* el filtro de vehículo
resultó **imposible** — `ConfigVehiculoId` NULL en el 100% de las 32
líneas Tarifario, confirmado como **bug de captura de CabinaFletes**
(no un hueco de la fase piloto, ver L-13). El filtro de peso sí se
aplicó: la mediana bajó de +30,6% a +9,1%.

*Paso 1e — la pregunta correcta no era "vs. la mediana", era "¿el
sistema hace lo que el código dice que hace?"* `PropuestaTarifaAdapter.CandidatoSapAsync`
no promedia — toma la tarifa SAP vigente **más alta** entre las que
aplican. Reproduciendo esa regla exacta, **20 de 32 líneas (62,5%)
coinciden con el valor exacto** cobrado en un primer intento.

*Paso 1g — SICE-TAC con el extracto real completo (no la muestra
curada):* se consiguió el extracto real del período 202608 (~2,1
millones de filas, el mismo cubo que usa el sistema real). Reproduciendo
`SeleccionarSice` (vehículo si hay dato → tipo de carga "General" si no
hay `tipologia` → carrocería si hay dato → prioridad vía estándar, menos
horas, menor valor), **las 12 de 12 líneas (100%) coinciden con el valor
exacto** cobrado.

*Paso 1h — Bogotá tiene 2 tablas de precio SAP paralelas.* De las 12
líneas SAP que no coincidían, se buscó el valor exacto sin restringir
zona (H-007 ya había confirmado que las 32 líneas SÍ tienen su valor
exacto en algún lado de `tarifa-plaza`) — apareció bajo
`ZTC_NORTE_BOGOTA` para 8 de esas 12 líneas. `tblZonasTarificacionRelacion`
confirma que son 2 tablas de precio paralelas para el área de Bogotá
(la ciudad en sí mapea a `ZTC_BOGOTA`; `ZTC_NORTE_BOGOTA` es para
satélites del norte como Chía/Zipaquirá, con precios sistemáticamente
más altos) — la explicación más probable es que el destino real de esas
2 cotizaciones fuera un municipio satélite del norte, simplificado a
"Bogotá" en `catalogo_rutas` (no confirmado con certeza).
**Sube a 28/32 (87,5%).**

*Paso 1i — cerrando las últimas 4 (Bucaramanga→Bogotá):* su valor exacto
aparece en una tarifa real de Devolución Contenedor (`lineanegocio="6"`,
la categoría excluida a propósito del resto de la comparación), pero
`lineas_solicitud` no tiene `RetornaContenedor=True` para estas líneas —
el contexto no cuadra. **El usuario confirmó que este valor se capturó
manualmente desde un Excel de referencia**, no lo generó el motor en
vivo — es captura manual (como `Fuente=Manual`, H-002), con un valor de
mercado real pero no comparable por el algoritmo de reproducción.

**Conclusión revisada de H-005:** las 32 líneas Tarifario y las 12
Sicetac quedan con una explicación — 28+12 por el algoritmo real de
cada motor, 4 por captura manual confirmada. Ninguna evidencia de que
CabinaFletes cotice mal. La pregunta de negocio real no es "¿por qué
cotiza distinto al mercado?", es **"¿esas políticas de selección (techo
en SAP, la más eficiente en SICE-TAC) son las que el negocio quiere?"**

**Mercado por su cuenta, sin depender de lo propio (§7.3, Pasos 2-4):**
- SAP por `LineaNegocio`: `Nacional` domina (59% del catálogo) y es la
  más barata en promedio; `Puerto a Interior`/`Interior a Puerto` casi
  idénticas entre sí (razonable, son la misma ruta en direcciones
  opuestas); `Licitaciones` tiene solo 1 fila en todo el catálogo.
- SAP por tipo de vehículo: patrón sensato — el precio sube con la
  capacidad (`Liviano 4 ton` la más barata, `Tractomula` la más cara).
  `Rígidos, Turbo o Sencillo` con solo 14 filas, muestra chica.
- SICE-TAC núcleo por tipo de carga: `Sin carga` la más barata,
  `Contenedor cargado`/`Granel líquido` las más caras — también sensato.
  Varias categorías con solo 6 filas cada una (recordar el diseño del
  núcleo: 6 filas por combinación).
- Salvedad transversal: estas dispersiones mezclan variación de negocio
  con variación de ruta — no se pueden separar sin el trabajo de la
  Fase 8.

**Categórica × categórica (§7.4, Paso 5):** `Modalidad` × líneas de
negocio SAP no aplica (Modalidad tiene solo 1 valor real). `Fuente` ×
`Resultado`: calculado (Tarifario y Sicetac ambos 50/50 Ganada/Perdida),
pero **declarado sin señal de negocio** — el usuario confirmó que
`Resultado` también lo marcó él mismo durante pruebas, no refleja una
decisión real de cliente. Se documenta como "no aplicable para inferir
negocio", no como patrón débil.

**Multivariado (§7.5):** no aplicable, tal como anticipaba el plan —
forzar clustering/PCA sobre unas pocas decenas de casos sería
sobreajustar a ruido.

**Qué le queda listo al siguiente notebook:** el hallazgo central de la
HU queda resuelto por completo en ambos lados — no es un gap de precio
a explicar, es el mecanismo de selección de cada motor (SAP y
SICE-TAC), una zona de Bogotá adicional, y una captura manual puntual,
todo confirmado con evidencia directa. Queda pendiente, aparte, extender
la verificación de SICE-TAC a los otros 26 períodos históricos
disponibles (`Historicos_SICE/`) para resolver L-07 a escala completa —
no se hizo en esta sesión por alcance/tamaño (cada archivo pesa
100-170MB). Todo es insumo directo para `hallazgos.md`/`limitaciones.md`.
Siguen las Fases 7-11 del plan (temporal, geográfico, outliers, sesgos,
vacíos) antes de la síntesis final.